# 01 - Data overview

Run the cells from top to bottom (Shift+Enter). This notebook only **reads** the prepared tables in `data/processed` (made by `scripts/01_prepare_data.py`); it does not change anything.

Tip: in the *Jupyter: Variables* panel, click the table icon next to a DataFrame to open it in Data Wrangler.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "config.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import config
from hfwnpo_aml import datasets
from hfwnpo_aml.survival import kaplan_meier, logrank_test, univariate_cox
pd.set_option("display.width", 160)
print("project:", ROOT)

## 1. Cohorts at a glance

In [ ]:
rows = []
for name in ["GSE114868", "GSE13159", "GSE9476", "GSE37642_U133AB", "GSE37642_U133P2", "TCGA_LAML", "BeatAML"]:
    if not datasets.available(name):
        rows.append({"cohort": name, "status": "missing"}); continue
    X, ph = datasets.load(name)
    r = {"cohort": name, "status": "ok", "samples": X.shape[0], "genes": X.shape[1],
         "PD-1 (PDCD1)": "PDCD1" in X, "PD-L1 (CD274)": "CD274" in X}
    if "label" in ph:
        r.update(task="diagnosis", AML=int(ph.label.sum()), controls=int((ph.label == 0).sum()))
    else:
        r.update(task="survival", deaths=int(ph.event.sum()), median_follow_up_months=round(ph.time.median(), 1))
    rows.append(r)
overview = pd.DataFrame(rows)
overview

## 2. PD-1 / PD-L1 expression: AML vs controls

In [ ]:
from scipy.stats import mannwhitneyu
cohorts = [c for c in ["GSE114868", "GSE13159", "GSE9476"] if datasets.available(c)]
fig, axes = plt.subplots(len(cohorts), 2, figsize=(7, 3 * len(cohorts)), squeeze=False)
for i, name in enumerate(cohorts):
    X, ph = datasets.load(name)
    for j, g in enumerate(["PDCD1", "CD274"]):
        ax = axes[i, j]
        if g not in X:
            ax.set_title(f"{name}: {g} not on platform"); ax.axis("off"); continue
        a, c = X.loc[ph.label == 1, g], X.loc[ph.label == 0, g]
        ax.boxplot([c, a], showfliers=False)
        ax.set_xticks([1, 2], [f"Control (n={len(c)})", f"AML (n={len(a)})"])
        ax.set_title(f"{name} - {g}  (p = {mannwhitneyu(a, c).pvalue:.2g})", fontsize=9)
plt.tight_layout(); plt.show()

## 3. PD-1 / PD-L1 and overall survival (median split)

In [ ]:
surv = [c for c in ["GSE37642_U133AB", "GSE37642_U133P2", "TCGA_LAML", "BeatAML"] if datasets.available(c)]
fig, axes = plt.subplots(len(surv), 2, figsize=(8, 3 * len(surv)), squeeze=False)
hr_rows = []
for i, name in enumerate(surv):
    X, ph = datasets.load(name)
    for j, g in enumerate(["PDCD1", "CD274"]):
        ax = axes[i, j]
        high = (X[g] > X[g].median()).values
        for grp, lab in ((~high, "low"), (high, "high")):
            t, s = kaplan_meier(ph.time.values[grp], ph.event.values[grp])
            ax.step(t, s, where="post", label=f"{g} {lab}")
        _, p = logrank_test(ph.time.values, ph.event.values, high.astype(float))
        ax.set_title(f"{name}: {g}  log-rank p = {p:.2g}", fontsize=9); ax.legend(fontsize=7)
        hr_rows.append({"cohort": name, "gene": g, **univariate_cox(X[g].values, ph.time.values, ph.event.values)})
plt.tight_layout(); plt.show()
pd.DataFrame(hr_rows).round(3)

## 4. BeatAML drug response (chemoresistance)

In [ ]:
d = datasets.load_drug()
if d is None:
    print("drug data not prepared")
else:
    Xd, auc = d
    print(auc.shape[0], "samples x", auc.shape[1], "inhibitors")
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
    for ax, drug in zip(axes, ["Cytarabine", "Venetoclax"]):
        ok = auc[drug].notna()
        ax.scatter(Xd.loc[ok, "CD274"], auc.loc[ok, drug], s=8, alpha=0.5)
        rho = Xd.loc[ok, "CD274"].corr(auc.loc[ok, drug], method="spearman")
        ax.set_xlabel("PD-L1 (CD274) expression"); ax.set_ylabel(f"{drug} AUC (higher = resistant)")
        ax.set_title(f"{drug}: n={ok.sum()}, Spearman rho={rho:.2f}", fontsize=9)
    plt.tight_layout(); plt.show()